# Practical Work · Unit 1 — Spam Detection with a Neural Network in PyTorch

Workshop module · ENSM · Dr. Amel Bouchemha

**Goal:** turn a handful of emails into numeric keyword vectors, then build, train and evaluate a small ANN in PyTorch, and finally study what happens when data is scarce.

| Step | What we do |
|---|---|
| 1 | Prepare the data (tokenise, 10-keyword 0/1 vectors, tensors) |
| 2 | Build the model (10 → 8 ReLU → 1 raw score) |
| 3 | Train (BCE-with-logits, Adam lr 0.01, 300 epochs, full batch) |
| 4 | Evaluate (test accuracy, per-email verdicts, `predict()`) |
| 5 | Five experiments, one change at a time, same seed |

**Requirements:** `pip install torch numpy`

In [1]:
import re
from dataclasses import dataclass

import torch
import torch.nn as nn

SEED = 42

## Step 1 · Prepare the data

Each email becomes a vector of 10 numbers: for every keyword, 1 if it appears in the content **or** the subject, 0 otherwise. A neural network cannot read text, so this vector is its input; the label (1 = spam, 0 = legitimate) is the expected output.

The `sender` addresses below are invented. They are only used by experiment 4.

In [2]:
@dataclass(frozen=True)
class Email:
    content: str
    subject: str
    label: int          # 1 = spam, 0 = legitimate
    sender: str = ""    # invented addresses, used by experiment 4


TRAIN_EMAILS = [
    # Spam
    Email("Get free gift cards now!", "Exclusive Offer", 1, "spam@example.com"),
    Email("You can win a free prize, click here", "Congratulations winner", 1, "promo@bigwin.net"),
    Email("Urgent: claim your prize now", "Final notice", 1, "alerts@secure-mail.com"),
    Email("Click to get a free vacation", "Limited offer", 1, "promo@travel-deals.com"),
    Email("Win cash today, click this link", "You won", 1, "spam@cashnow.org"),
    Email("Special offer: cheap watches, buy now", "Huge discount", 1, "sales@watchshop.com"),
    Email("Urgent action required, click to verify your account", "Account suspended", 1, "support@acct-verify.net"),
    Email("Free gift for you, claim now", "Exclusive gift", 1, "gifts@promo-gifts.com"),
    # Legitimate
    Email("Hi team, the meeting is moved to 3pm", "Meeting update", 0, "manager@company.com"),
    Email("Please find attached the monthly report", "Monthly report", 0, "finance@company.com"),
    Email("Thanks for your help with the project", "Thanks", 0, "sara@company.com"),
    Email("Can we meet tomorrow to discuss the report?", "Quick question", 0, "ahmed@company.com"),
    Email("Lunch at noon? Let me know", "Lunch", 0, "karim@company.com"),
    Email("The free parking is on the left side of the building", "Parking info", 0, "admin@company.com"),
    Email("Attached is the job offer letter, please sign it", "Job offer", 0, "hr@company.com"),
    Email("Thanks for the update, see you at the meeting", "Re: update", 0, "manager@company.com"),
]

# Test set: never used for training
TEST_EMAILS = [
    Email("Win a free iPhone, click now", "Prize draw", 1, "promo@iphone-win.com"),
    Email("Limited offer, free shipping on everything, click", "Big sale", 1, "sales@megastore.com"),
    Email("Here are the notes from yesterday's class", "Class notes", 0, "prof@university.edu"),
    Email("Reminder: project report due Friday", "Reminder", 0, "manager@company.com"),
]

# Experiment 2: three extra training emails (two legitimate ones contain "free" / "offer")
EXTRA_TRAIN_EMAILS = [
    Email("Win a free cruise, click to claim your prize", "Final offer", 1, "promo@cruise-deals.com"),
    Email("Are you free on Thursday to review the draft?", "Draft review", 0, "sara@company.com"),
    Email("They accepted our offer, thanks for the support", "Good news", 0, "ahmed@company.com"),
]

# Keywords (keep this order)
KEYWORDS = ["free", "win", "offer", "gift", "urgent",
            "click", "prize", "meeting", "report", "thanks"]

In [3]:
def tokenize(text):
    """Lowercase, whole words only, no punctuation."""
    return re.findall(r"[a-z]+", text.lower())


def to_vector(email, keywords=KEYWORDS, mode="binary", use_sender=False):
    """Feature vector of one email (content + subject).

    mode="binary": 1 if the keyword is present, else 0.
    mode="count":  number of occurrences of the keyword.
    use_sender:    append 1 if the address contains "spam" or "promo".
    """
    tokens = tokenize(f"{email.content} {email.subject}")
    if mode == "binary":
        present = set(tokens)
        vec = [1.0 if kw in present else 0.0 for kw in keywords]
    elif mode == "count":
        vec = [float(tokens.count(kw)) for kw in keywords]
    else:
        raise ValueError("mode must be 'binary' or 'count'")
    if use_sender:
        s = email.sender.lower()
        vec.append(1.0 if ("spam" in s or "promo" in s) else 0.0)
    return vec


def build_tensors(emails, **kwargs):
    X = torch.tensor([to_vector(e, **kwargs) for e in emails])
    y = torch.tensor([float(e.label) for e in emails])
    return X, y


X_train, y_train = build_tensors(TRAIN_EMAILS)
X_test, y_test = build_tensors(TEST_EMAILS)

# Checks from the handout
print("Email 1 ->", X_train[0].int().tolist())   # expected [1,0,1,1,0,0,0,0,0,0]
print("T1      ->", X_test[0].int().tolist())    # expected [1,1,0,0,0,1,1,0,0,0]
print("X_train:", tuple(X_train.shape))          # expected (16, 10)
print("X_test :", tuple(X_test.shape))           # expected (4, 10)

Email 1 -> [1, 0, 1, 1, 0, 0, 0, 0, 0, 0]
T1      -> [1, 1, 0, 0, 0, 1, 1, 0, 0, 0]
X_train: (16, 10)
X_test : (4, 10)


## Step 2 · Build the model

A class inheriting from `nn.Module`: **10 inputs → 8 hidden neurons (ReLU) → 1 output**. The output is a *raw score* (a logit): there is no activation on the last layer, because the loss we use in step 3 applies the sigmoid internally.

In [4]:
class SpamANN(nn.Module):
    def __init__(self, input_size=10, hidden=8):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)   # no activation on the output

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x))).squeeze(-1)


torch.manual_seed(SEED)                   # fixed seed -> reproducible weights
model = SpamANN(input_size=len(KEYWORDS))

# Check: passing X_train returns 16 values
print("Output shape:", tuple(model(X_train).shape))   # expected (16,)

Output shape: (16,)


## Step 3 · Train the model

* Loss: `BCEWithLogitsLoss` (binary classification on raw scores)
* Optimiser: Adam, learning rate 0.01
* 300 epochs, the whole training set as one batch
* Loss and training accuracy printed every 50 epochs

The loss should decrease over the epochs.

In [5]:
def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X))


def predict_labels(model, X):
    return (predict_proba(model, X) > 0.5).float()


def accuracy(model, X, y):
    """Share of correct emails (0.5 threshold on the probability)."""
    return (predict_labels(model, X) == y).float().mean().item()


def train_model(model, X, y, epochs=300, lr=0.01, log_every=50):
    """BCEWithLogitsLoss + Adam, whole training set as one batch."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        if log_every and epoch % log_every == 0:
            print(f"Epoch {epoch:>3} | loss {loss.item():.4f} | "
                  f"train acc {accuracy(model, X, y):.2%}")


train_model(model, X_train, y_train)

Epoch  50 | loss 0.2065 | train acc 93.75%
Epoch 100 | loss 0.1010 | train acc 93.75%
Epoch 150 | loss 0.0919 | train acc 93.75%
Epoch 200 | loss 0.0893 | train acc 93.75%
Epoch 250 | loss 0.0884 | train acc 93.75%
Epoch 300 | loss 0.0879 | train acc 93.75%


## Step 4 · Evaluate

Test accuracy, then for each test email: the spam probability, the true label and the verdict. Finally `predict(content, subject)` lets you try the model on your own emails.

In [6]:
baseline_test_acc = accuracy(model, X_test, y_test)
print(f"Test accuracy: {baseline_test_acc:.2%}\n")

probs = predict_proba(model, X_test)
print(f"{'No.':<4}{'Subject':<14}{'P(spam)':>9}{'True':>6}  Verdict")
for i, (e, p) in enumerate(zip(TEST_EMAILS, probs), 1):
    is_spam = p.item() > 0.5
    status = "correct" if is_spam == bool(e.label) else "WRONG"
    print(f"T{i:<3}{e.subject:<14}{p.item():>9.3f}{e.label:>6}  "
          f"{'SPAM' if is_spam else 'LEGIT'} ({status})")

Test accuracy: 100.00%

No. Subject         P(spam)  True  Verdict
T1  Prize draw        1.000     1  SPAM (correct)
T2  Big sale          0.999     1  SPAM (correct)
T3  Class notes       0.010     0  LEGIT (correct)
T4  Reminder          0.000     0  LEGIT (correct)


In [7]:
def predict(content, subject=""):
    """Spam probability of a new email (baseline features)."""
    x = torch.tensor([to_vector(Email(content, subject, 0))])
    return predict_proba(model, x).item()


# Two emails of your own: one obvious spam, one legitimate
print("Obvious spam : P(spam) =", f"{predict('Claim your free prize now, click here', 'You won'):.3f}")
print("Legitimate   : P(spam) =", f"{predict('Please send me the report before the meeting', 'Report'):.3f}")

Obvious spam : P(spam) = 1.000
Legitimate   : P(spam) = 0.000


## Step 5 · Experiments

Change **one thing at a time** and keep the same random seed so results stay comparable. `run_experiment` re-seeds before building each model, so the baseline row reproduces step 4 exactly.

1. Add 3 keywords of your choice (here: `cash`, `claim`, `verify`)
2. Add 3 training emails, including a legitimate one containing "free" or "offer"
3. Replace presence (0/1) with the number of occurrences of each word
4. Add a sender feature: 1 if the address contains "spam" or "promo"
5. Remove "free" from the keywords and identify which emails become misclassified

In [8]:
def run_experiment(keywords=KEYWORDS, mode="binary", use_sender=False,
                   train_emails=TRAIN_EMAILS, test_emails=TEST_EMAILS):
    torch.manual_seed(SEED)   # same seed -> comparable results
    opts = dict(keywords=keywords, mode=mode, use_sender=use_sender)
    Xtr, ytr = build_tensors(train_emails, **opts)
    Xte, yte = build_tensors(test_emails, **opts)

    m = SpamANN(input_size=Xtr.shape[1])
    train_model(m, Xtr, ytr, log_every=None)

    def wrong(emails, X, y):
        preds = predict_labels(m, X)
        return [e.subject for e, p, t in zip(emails, preds, y) if p != t]

    return {
        "train_acc": accuracy(m, Xtr, ytr),
        "test_acc": accuracy(m, Xte, yte),
        "wrong_train": wrong(train_emails, Xtr, ytr),
        "wrong_test": wrong(test_emails, Xte, yte),
    }


EXPERIMENTS = {
    "Baseline": {},
    "1. Add 3 keywords (cash, claim, verify)":
        dict(keywords=KEYWORDS + ["cash", "claim", "verify"]),
    "2. Add 3 training emails":
        dict(train_emails=TRAIN_EMAILS + EXTRA_TRAIN_EMAILS),
    "3. Word counts instead of 0/1":
        dict(mode="count"),
    "4. Sender feature":
        dict(use_sender=True),
    "5. Remove 'free'":
        dict(keywords=[k for k in KEYWORDS if k != "free"]),
}

results = {name: run_experiment(**cfg) for name, cfg in EXPERIMENTS.items()}
base = results["Baseline"]["test_acc"]

print(f"{'Experiment':<42}{'Train':>7}{'Test':>7}{'Change':>8}")
for name, r in results.items():
    print(f"{name:<42}{r['train_acc']:>7.0%}{r['test_acc']:>7.0%}{r['test_acc'] - base:>+8.0%}")

print()
for name, r in results.items():
    print(name)
    print("   misclassified (train):", ", ".join(r["wrong_train"]) or "-")
    print("   misclassified (test) :", ", ".join(r["wrong_test"]) or "-")

Experiment                                  Train   Test  Change
Baseline                                      94%   100%     +0%
1. Add 3 keywords (cash, claim, verify)       94%   100%     +0%
2. Add 3 training emails                      95%   100%     +0%
3. Word counts instead of 0/1                100%   100%     +0%
4. Sender feature                             94%   100%     +0%
5. Remove 'free'                              94%   100%     +0%

Baseline
   misclassified (train): Job offer
   misclassified (test) : -
1. Add 3 keywords (cash, claim, verify)
   misclassified (train): Job offer
   misclassified (test) : -
2. Add 3 training emails
   misclassified (train): Huge discount
   misclassified (test) : -
3. Word counts instead of 0/1
   misclassified (train): -
   misclassified (test) : -
4. Sender feature
   misclassified (train): Huge discount
   misclassified (test) : -
5. Remove 'free'
   misclassified (train): Huge discount
   misclassified (test) : -


## Observations

*Fill this in after running the notebook.*

* **Baseline test accuracy:** …
* **Which experiments changed the test accuracy, and why?** …
* **Experiment 5:** which emails became misclassified once "free" was removed? …
* **Limits:** with only 4 test emails, accuracy moves in steps of 25 %, so a single email changes the result a lot. These comparisons illustrate the ideas; they are not statistically meaningful.